# Week 2 — Cameras & IPM

**1-camera IPM is required before the 3-camera stitch.** Student fills live in the module `.py` files.

Synthetic calibrated frames are checked in under `data/m01_sample` (CC0), not captured from a real vehicle rig.

## Session card

**Today's win:** Warp one camera to the ground plane, then measure how pitch error grows with range.

**Time:** 25 / 55 / 90 minutes

A day counts when you export an artifact or pass the tests for a fill you wrote. Opening the notebook does not. Set pause_week to true if you need a week off; the count stays where it is.

**Your stack so far**

- [ ] m00 — Driving ML Gym
- [ ] m01 — Cameras & IPM
- [ ] m02 — HydraNet
- [ ] m03 — BEV transform
- [ ] m04 — Occupancy
- [ ] m05 — Vector tracking
- [ ] m06 — Planning
- [ ] m07 — Control
- [ ] m08 — Capstone
- [ ] m09 — System architecture

XP is not awarded for opening this notebook.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / 'modules' / '01_camera_geometry').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
from common.progress import format_stack

progress_path = repo / 'artifacts' / 'progress.json'
if progress_path.is_file():
    import json
    with progress_path.open() as f:
        progress = json.load(f)
    print(format_stack(progress))
else:
    print('No progress file yet. Opening this notebook awards 0 XP.')

**Principle 1 — Pinhole projection.**

A pinhole maps a 3D ray to a pixel by similar triangles. A point at camera coordinates `(X, Y, Z)` hits the sensor at `x = f X / Z`, `y = f Y / Z`. Depth is in the denominator, so far objects shrink. Pixel coordinates add the principal point: `u = fx X/Z + cx`. `K` packs `fx`, `fy`, `cx`, `cy` into a 3×3 matrix. You will build `K` yourself.

**Principle 2 — Homography on the road plane.**

A homography is a 3×3 map between two planes. The road is the plane `Z=0` in the ego frame (X forward, Y left, Z up). A camera point is `P_cam = R P_ego + t`. Expanding `R` into columns `r1`, `r2`, `r3`, the `r3` term is multiplied by `Z`. On the road `Z` is 0, so `r3` drops out and `H = K [r1 r2 t]` maps `(X, Y, 1)` to homogeneous pixels. You will build that `H`. Using `R = I` on a pitched camera is the wrong plane-to-image map.

**Principle 3 — Extrinsics set meters on the ground.**

Meters on the ground come from extrinsics, not from the network. If pitch is wrong, the ray from a pixel hits `Z=0` in the wrong place. The same angular error covers more ground farther away, so the meter error grows with range. A 2° pitch bias that looks small in the image is a large longitudinal error at 40 m. You will measure that, not memorize a slogan.

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / 'modules' / '01_camera_geometry').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules' / '01_camera_geometry'))
data = repo / 'data' / 'm01_sample'

from camera_model import PinholeCamera, build_intrinsic_matrix
from calibrate_rig import build_tesla_style_rig
from ipm import IPMTransformer, build_ground_homography
from stitch import stitch_three_cameras
from pitch_sensitivity import pitch_shift_meters

with open(data / 'calib.json') as f:
    calib = json.load(f)
frames = {k: cv2.imread(str(data / f'{k}.png')) for k in ('front', 'left', 'right')}
for k, v in frames.items():
    plt.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    plt.title(k)
    plt.axis('off')
    plt.show()

**Similar triangles.** Pick focal length `f = 400` px and lateral offset `X = 10` m. Compute the image coordinate `x = f X / Z` at `Z = 20` m and at `Z = 40` m.

In [ ]:
f, X = 400.0, 10.0
x_20 = f * X / 20.0
x_40 = f * X / 40.0
print('x at Z=20 m:', x_20)
print('x at Z=40 m:', x_40)
print('Doubling depth halves the image coordinate — the pinhole principle.')

**FILL — `build_intrinsic_matrix(fx, fy, cx, cy) -> (3, 3)`**

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

Implement it in `modules/01_camera_geometry/camera_model.py`, not in this notebook.

In [ ]:
try:
    K = build_intrinsic_matrix(calib['fx'], calib['fy'], calib['cx'], calib['cy'])
    assert abs(K[0, 0] - calib['fx']) < 1e-6
    assert abs(K[1, 1] - calib['fy']) < 1e-6
    assert abs(K[0, 2] - calib['cx']) < 1e-6
    assert abs(K[2, 2] - 1.0) < 1e-6
    print('K ok\n', K)
except NotImplementedError:
    print('STOP: implement build_intrinsic_matrix in camera_model.py')

In [ ]:
cams = build_tesla_style_rig(calib['width'], calib['height'])
front = cams['front']
for pt in [(10, 0, 0), (20, 1.5, 0), (30, -1.5, 0)]:
    pix, ok = front.project_ego_to_pixel([pt])
    print(pt, '->', pix[0], 'valid', ok[0])

**Extrinsics: ego → camera.** `P_cam = R P_ego + T`. `R` rotates ego axes into the camera frame; `T` is the translation column (same convention as `PinholeCamera`).

In [ ]:
print('front.R (rounded):\n', np.round(front.R, 3))
print('front.T.ravel():', front.T.ravel())
print('calib pitch_deg:', calib['pitch_deg'])
print('Pitch is not zero, so R is not a pure axis swap.')

**Derive the ground homography.** On the road plane `Z = 0`, the third column of the extrinsic block is unused:

$$H = K \,[r_1 \; r_2 \; t]$$

`r3` is the column multiplied by `Z`, hence absent on the road.

**FILL — `build_ground_homography(K, R, t)`** in `modules/01_camera_geometry/ipm.py`. Do not paste the solution here.

In [ ]:
try:
    H = build_ground_homography(front.K, front.R, front.T)
    pt = np.array([15.0, 0.0, 1.0])
    uv = H @ pt
    uv = uv[:2] / uv[2]
    pix, _ = front.project_ego_to_pixel([[15, 0, 0]])
    err_h = np.max(np.abs(uv - pix[0]))
    print('H pixel', uv, 'project', pix[0], 'max abs err', err_h)
    assert err_h < 1e-2
    H_wrong = build_ground_homography(front.K, np.eye(3), front.T)
    uv_w = H_wrong @ pt
    uv_w = uv_w[:2] / uv_w[2]
    err_wrong = np.linalg.norm(uv_w - pix[0])
    print('R=I error px', err_wrong)
    assert err_wrong > 5.0
except NotImplementedError:
    print('STOP: implement build_ground_homography in ipm.py')

In [ ]:
ipm = IPMTransformer(front, x_range=(4, 40), y_range=(-10, 10), bev_resolution=0.1)
bev = ipm.warp_to_bev(frames['front'])
plt.imshow(cv2.cvtColor(bev, cv2.COLOR_BGR2RGB))
plt.title('1-cam IPM')
plt.axis('off')
plt.show()

In [ ]:
stitched = stitch_three_cameras(frames, cams)
plt.imshow(cv2.cvtColor(stitched, cv2.COLOR_BGR2RGB))
plt.title('3-cam stitch')
plt.axis('off')
plt.show()

**FROM SCRATCH — `pitch_shift_meters(calib, delta_deg, range_m) -> float`** in `modules/01_camera_geometry/pitch_sensitivity.py`.

Add `delta_deg` to the calib pitch, then return estimated forward range minus true range for the ground point `(range_m, 0, 0)`. `delta_deg = 0` should be ~0; absolute error should grow with range. Implement the function in the `.py` file — do not spell out the algorithm in this notebook.

In [ ]:
from extrinsics import create_euler_rotation, camera_position_to_translation

ranges = [10, 20, 40]
delta_deg = 2.0

def _scaffold_pitch_shift(rng, delta):
    pix, _ = front.project_ego_to_pixel([[rng, 0, 0]])
    R2 = create_euler_rotation(calib['pitch_deg'] + delta, 0, 0)
    T2 = camera_position_to_translation(R2, np.array(calib['cam_position_ego']))
    cam2 = PinholeCamera(
        'biased', front.fx, front.fy, front.cx, front.cy,
        front.width, front.height, R2, T2,
    )
    est, _ = cam2.project_pixels_to_ground(pix)
    return float(est[0, 0] - rng)

try:
    shifts = [pitch_shift_meters(calib, delta_deg, r) for r in ranges]
    for r, s in zip(ranges, shifts):
        print(f'range {r} m shift {s:.4f} m')
    plt.plot(ranges, shifts, marker='o')
    plt.xlabel('range (m)')
    plt.ylabel('forward shift (m)')
    plt.title('pitch_shift_meters (student fill)')
    plt.grid(True, alpha=0.3)
    plt.show()
    assert abs(shifts[2]) > abs(shifts[0])
except NotImplementedError:
    print('STOP: implement pitch_shift_meters in pitch_sensitivity.py')
    scaffold = [_scaffold_pitch_shift(r, delta_deg) for r in ranges]
    for r, s in zip(ranges, scaffold):
        print(f'[scaffold] range {r} m shift {s:.4f} m')
    plt.plot(ranges, scaffold, marker='o', linestyle='--')
    plt.xlabel('range (m)')
    plt.ylabel('forward shift (m)')
    plt.title('Reference scaffold (not student pitch_shift_meters)')
    plt.grid(True, alpha=0.3)
    plt.show()

**Free response:** Which principle explains the smear when pitch is wrong? Why does the meter error grow with range?

_Write 3–6 sentences here._

**Assignment:** The rig's pitch is 2° off the truth. What would you change so a lane point at 25 m lands back on the correct ground coordinate? See `python modules/01_camera_geometry/break_it_fix_it.py`.

_Write 3–6 sentences here._

In [ ]:
from calibrate_rig import main as calib_main
m01_metrics = calib_main()
print('metrics outputs:', m01_metrics.get('outputs', {}))
print('run_id:', m01_metrics.get('run_id'))

**Tests**

```bash
python3 -m pytest modules/01_camera_geometry -q
```

Scaffold tests always run. `test_assignment_solutions.py` imports `solutions/01_camera_geometry`; student fill tests skip until implemented and fail if the implementation is wrong.

**Come back cue**

Tomorrow: 25-min pitch check — implement pitch_shift_meters and plot 10 m, 20 m, and 40 m.

Suggested slot: 25 minutes. 55 or 90 if you are also writing the principle cells.

In [ ]:
import sys
import json
from pathlib import Path

repo = Path.cwd()
if not (repo / 'modules' / '01_camera_geometry').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'modules'))
from common.progress import come_back_cue

print(come_back_cue('m01'))
progress_path = repo / 'artifacts' / 'progress.json'
if progress_path.is_file():
    with progress_path.open() as f:
        xp = json.load(f).get('xp', 0)
    print(f'XP so far: {xp}')